# Loan Approval Model Training

This notebook loads and validates the loan dataset, creates a preprocessing pipeline, trains a random-forest classifier, evaluates it on an untouched holdout set, and saves the complete model. Run the cells from top to bottom.

In [1]:
# 1. Import the required libraries
import json
import warnings
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Configure paths and training settings

Change `TEST_SIZE`, `CV_FOLDS`, or `RANDOM_STATE` here if you want to experiment. The notebook expects to be run from the project folder.

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "loan_approval_dataset.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "loan_approval_model.joblib"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics.json"
PIPELINE_CACHE = PROJECT_ROOT / ".pipeline_cache"

TEST_SIZE = 0.20
CV_FOLDS = 5
RANDOM_STATE = 42

TARGET_COLUMN = "loan_status"
ID_COLUMN = "loan_id"
LABELS = ["Rejected", "Approved"]

NUMERIC_FEATURES = [
    "no_of_dependents",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]
CATEGORICAL_FEATURES = ["education", "self_employed"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
ASSET_FEATURES = [
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Open the project folder in VS Code first."
    )

print(f"Project folder: {PROJECT_ROOT}")

Project folder: d:\Machine Learnig Projects\ML_service(Loan Prediction)


## 3. Load and clean the dataset

The CSV contains spaces after its commas, so `skipinitialspace=True` and string trimming normalize the column names and categorical values.

In [3]:
data = pd.read_csv(DATA_PATH, skipinitialspace=True)
data.columns = data.columns.str.strip()

required_columns = set(FEATURES + [ID_COLUMN, TARGET_COLUMN])
missing_columns = sorted(required_columns.difference(data.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

data = data[FEATURES + [ID_COLUMN, TARGET_COLUMN]].copy()

for column in CATEGORICAL_FEATURES + [TARGET_COLUMN]:
    data[column] = data[column].astype("string").str.strip()

for column in NUMERIC_FEATURES + [ID_COLUMN]:
    data[column] = pd.to_numeric(data[column], errors="raise")

print(f"Dataset shape: {data.shape}")
display(data.head())

Dataset shape: (4269, 13)


,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,education,self_employed,loan_id,loan_status
0,2,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Graduate,No,1,Approved
1,0,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Not Graduate,Yes,2,Rejected
2,3,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Graduate,No,3,Rejected
3,3,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Graduate,No,4,Rejected
4,5,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Not Graduate,Yes,5,Rejected


## 4. Validate and explore the data

This checks the IDs, missing data, target labels, category values, duplicates, class balance, and numerical ranges. The 28 negative residential-asset entries are reported but retained so the notebook does not silently rewrite the source data.

In [4]:
if data.empty:
    raise ValueError("Dataset contains no rows.")
if data[ID_COLUMN].isna().any():
    raise ValueError(f"{ID_COLUMN} contains missing values.")
if data[ID_COLUMN].duplicated().any():
    raise ValueError(f"{ID_COLUMN} must be unique.")
if data[FEATURES + [TARGET_COLUMN]].isna().any().any():
    missing_values = data[FEATURES + [TARGET_COLUMN]].isna().sum()
    missing_values = missing_values[missing_values.gt(0)].to_dict()
    raise ValueError(f"Dataset contains missing values: {missing_values}")

observed_labels = set(data[TARGET_COLUMN].unique())
if observed_labels != set(LABELS):
    raise ValueError(f"Expected target labels {LABELS}; found {sorted(observed_labels)}")

allowed_categories = {
    "education": {"Graduate", "Not Graduate"},
    "self_employed": {"Yes", "No"},
}
for column, allowed_values in allowed_categories.items():
    unexpected = sorted(set(data[column].unique()).difference(allowed_values))
    if unexpected:
        raise ValueError(f"Unexpected values in {column}: {unexpected}")

negative_assets = {
    column: int(data[column].lt(0).sum())
    for column in ASSET_FEATURES
    if data[column].lt(0).any()
}
if negative_assets:
    warnings.warn(f"Negative asset values retained: {negative_assets}")

target_counts = data[TARGET_COLUMN].value_counts()
print(f"Missing values: {int(data.isna().sum().sum())}")
print(f"Duplicate rows: {int(data.duplicated().sum())}")
display(pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(data) * 100).round(2),
}))
display(data[NUMERIC_FEATURES].describe().T)

Missing values: 0
Duplicate rows: 0


C:\Users\MSI\AppData\Local\Temp\ipykernel_21260\4000406094.py:31: UserWarning: Negative asset values retained: {'residential_assets_value': 28}
  warnings.warn(f"Negative asset values retained: {negative_assets}")


,count,percentage
loan_status,,
Approved,2656,62.22
Rejected,1613,37.78


,count,mean,std,min,25%,50%,75%,max
no_of_dependents,4269.0,2.498712e+00,1.695910e+00,0.0,1.0,3.0,4.0,5.0
income_annum,4269.0,5.059124e+06,2.806840e+06,200000.0,2700000.0,5100000.0,7500000.0,9900000.0
loan_amount,4269.0,1.513345e+07,9.043363e+06,300000.0,7700000.0,14500000.0,21500000.0,39500000.0
loan_term,4269.0,1.090045e+01,5.709187e+00,2.0,6.0,10.0,16.0,20.0
cibil_score,4269.0,5.999361e+02,1.724304e+02,300.0,453.0,600.0,748.0,900.0
residential_assets_value,4269.0,7.472617e+06,6.503637e+06,-100000.0,2200000.0,5600000.0,11300000.0,29100000.0
commercial_assets_value,4269.0,4.973155e+06,4.388966e+06,0.0,1300000.0,3700000.0,7600000.0,19400000.0
luxury_assets_value,4269.0,1.512631e+07,9.103754e+06,300000.0,7500000.0,14600000.0,21700000.0,39200000.0
bank_asset_value,4269.0,4.976692e+06,3.250185e+06,0.0,2300000.0,4600000.0,7100000.0,14700000.0


## 5. Select features and create a holdout set

`loan_id` is excluded because it identifies a record rather than describing an applicant. Stratification preserves the Approved/Rejected ratio in both partitions.

In [5]:
X = data[FEATURES]
y = data[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Holdout test rows: {len(X_test)}")

Training rows: 3415
Holdout test rows: 854


## 6. Build preprocessing and model pipelines

Numerical values use median imputation. Categorical values use the most frequent category and one-hot encoding. The preprocessing and classifier are combined so the saved model accepts raw applicant rows.

In [6]:
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

classifier = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ],
    memory=str(PIPELINE_CACHE),
)

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, 

## 7. Cross-validate on the training partition

Five-fold validation estimates how consistently the model performs without looking at the final holdout set.

In [7]:
cross_validator = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cross_validator,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "roc_auc": "roc_auc",
    },
    n_jobs=1,
)

cv_summary = pd.DataFrame({
    metric: {
        "mean": cv_scores[f"test_{metric}"].mean(),
        "standard_deviation": cv_scores[f"test_{metric}"].std(),
    }
    for metric in ("accuracy", "macro_f1", "roc_auc")
}).T
display(cv_summary.round(4))

,mean,standard_deviation
accuracy,0.9777,0.0067
macro_f1,0.9764,0.0071
roc_auc,0.9978,0.0007


## 8. Train and evaluate on the untouched holdout set

Accuracy shows total correctness, macro-F1 weights both classes equally, and ROC-AUC measures ranking quality across probability thresholds.

In [8]:
model.fit(X_train, y_train)
fitted_classifier = model.named_steps["classifier"]
predictions = model.predict(X_test)

class_names = list(model.classes_)
approved_index = class_names.index("Approved")
approval_probabilities = model.predict_proba(X_test)[:, approved_index]
binary_test_target = y_test.eq("Approved").astype(int)

accuracy = accuracy_score(y_test, predictions)
balanced_accuracy = balanced_accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average="macro")
roc_auc = roc_auc_score(binary_test_target, approval_probabilities)
confusion = confusion_matrix(y_test, predictions, labels=LABELS)
report = classification_report(
    y_test, predictions, labels=LABELS, output_dict=True, zero_division=0
)

display(pd.Series({
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "macro_f1": macro_f1,
    "roc_auc": roc_auc,
}, name="holdout_score").round(4))

display(pd.DataFrame(confusion, index=LABELS, columns=LABELS))
print(classification_report(
    y_test, predictions, labels=LABELS, digits=4, zero_division=0
))

accuracy             0.9742
balanced_accuracy    0.9732
macro_f1             0.9726
roc_auc              0.9979
Name: holdout_score, dtype: float64

,Rejected,Approved
Rejected,313,10
Approved,12,519


              precision    recall  f1-score   support

    Rejected     0.9631    0.9690    0.9660       323
    Approved     0.9811    0.9774    0.9792       531

    accuracy                         0.9742       854
   macro avg     0.9721    0.9732    0.9726       854
weighted avg     0.9743    0.9742    0.9743       854



## 9. Inspect feature importance

Random-forest importance estimates how much each encoded feature contributes to reducing prediction error within its trees.

In [9]:
transformed_features = model.named_steps["preprocessor"].get_feature_names_out()
feature_importance = sorted(
    zip(transformed_features, fitted_classifier.feature_importances_, strict=True),
    key=lambda item: item[1],
    reverse=True,
)

importance_table = pd.DataFrame(
    feature_importance, columns=["feature", "importance"]
).head(10)
display(importance_table)

,feature,importance
0,numeric__cibil_score,0.828042
1,numeric__loan_term,0.050629
2,numeric__loan_amount,0.027556
3,numeric__luxury_assets_value,0.016801
4,numeric__commercial_assets_value,0.016446
5,numeric__income_annum,0.016230
6,numeric__residential_assets_value,0.015427
7,numeric__bank_asset_value,0.014191
8,numeric__no_of_dependents,0.007692
9,categorical__self_employed_No,0.001937


## 10. Save the model and metrics

The `.joblib` file stores both preprocessing and prediction. The JSON file stores the data profile and evaluation results.

In [10]:
cross_validation_metrics = {
    "folds": CV_FOLDS,
    **{
        metric: {
            "mean": float(cv_scores[f"test_{metric}"].mean()),
            "standard_deviation": float(cv_scores[f"test_{metric}"].std()),
        }
        for metric in ("accuracy", "macro_f1", "roc_auc")
    },
}

metrics = {
    "data": {
        "rows": len(data),
        "input_features": len(FEATURES),
        "duplicate_rows": int(data.duplicated().sum()),
        "target_distribution": {
            str(label): int(count) for label, count in target_counts.items()
        },
        "negative_asset_values": negative_assets,
    },
    "split": {
        "training_rows": len(X_train),
        "test_rows": len(X_test),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
    },
    "model": {
        "type": "RandomForestClassifier",
        "features": FEATURES,
        "target": TARGET_COLUMN,
        "classes": class_names,
    },
    "cross_validation_on_training_partition": cross_validation_metrics,
    "holdout_test": {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_f1": float(macro_f1),
        "roc_auc": float(roc_auc),
        "confusion_matrix": {
            "label_order": LABELS,
            "values": confusion.tolist(),
        },
        "classification_report": report,
    },
    "top_feature_importance": [
        {"feature": str(feature), "importance": float(importance)}
        for feature, importance in feature_importance[:10]
    ],
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Model saved to: {MODEL_PATH}")
print(f"Metrics saved to: {METRICS_PATH}")

Model saved to: d:\Machine Learnig Projects\ML_service(Loan Prediction)\models\loan_approval_model.joblib
Metrics saved to: d:\Machine Learnig Projects\ML_service(Loan Prediction)\models\metrics.json


## 11. Try one prediction

This final optional cell reloads the saved artifact and predicts one example applicant.

In [11]:
saved_model = joblib.load(MODEL_PATH)
example_applicant = pd.DataFrame([{
    "no_of_dependents": 2,
    "education": "Graduate",
    "self_employed": "No",
    "income_annum": 5_000_000,
    "loan_amount": 12_000_000,
    "loan_term": 10,
    "cibil_score": 750,
    "residential_assets_value": 5_000_000,
    "commercial_assets_value": 2_000_000,
    "luxury_assets_value": 8_000_000,
    "bank_asset_value": 3_000_000,
}])

predicted_label = saved_model.predict(example_applicant)[0]
approved_index = list(saved_model.classes_).index("Approved")
approval_probability = saved_model.predict_proba(example_applicant)[0, approved_index]

print(f"Prediction: {predicted_label}")
print(f"Approval probability: {approval_probability:.2%}")

Prediction: Approved
Approval probability: 98.43%
